# NeuroFetal AI — Calibration & Ensemble Retraining

**Version 5.0** — Calibration Fix Phase

This notebook retrains the diverse ensemble with calibration and uncertainty fixes:

### What Changed
| # | Fix | File | Effect |
|---|-----|------|--------|
| 1 | Isotonic Regression on meta-learner | `train_diverse_ensemble.py` | Well-calibrated probabilities |
| 2 | Ensemble variance uncertainty | `evaluate_uncertainty.py` | Uncertainty = std(ResNet, Inception, XGB) |
| 3 | Focal Loss γ reduced (2.0 → 1.5) | `focal_loss.py` | Less overconfident predictions |
| 4 | Ensemble inference in app | `app.py` + `model_loader.py` | Live uncertainty from 3 models |

### Pipeline Steps
| # | Phase | Script |
|---|-------|--------|
| 1 | Setup | Clone repo, install deps |
| 2 | Data Ingestion | `data_ingestion.py` |
| 3 | SSL Pretraining | `pretrain.py` |
| 4 | Primary Training (γ=1.5) | `train.py` |
| 5 | Ensemble Training + Isotonic Calibration | `train_diverse_ensemble.py` |
| 6 | Uncertainty Evaluation | `evaluate_uncertainty.py` |
| 7 | Push Models to GitHub | `git push` |

## 1. Setup Environment

In [2]:
from google.colab import userdata
import os

# 1. GitHub Authentication
GITHUB_REPO = "Krishna200608/NeuroFetal-AI"
+++
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✓ GitHub Token loaded from Secrets.")
except Exception as e:
    print("⚠️ Error loading GITHUB_TOKEN from Secrets. Falling back to manual input.")
    from getpass import getpass
    GITHUB_TOKEN = getpass("Enter GitHub Personal Access Token (PAT): ")

os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
os.environ['GITHUB_REPO'] = GITHUB_REPO

✓ GitHub Token loaded from Secrets.


In [3]:
# 2. Clone Repository & Checkout main branch
import shutil
import os

# Reset to /content before deleting the repo folder
try:
    os.chdir("/content")
except:
    pass

# Clean up any previous clone
if os.path.exists("/content/NeuroFetal-AI"):
    shutil.rmtree("/content/NeuroFetal-AI")

print("Cloning repository...")
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git

os.chdir("/content/NeuroFetal-AI")

# Checkout main branch
!git checkout main
!git pull origin main
print("✓ Cloned and checked out main!")

Cloning repository...
Cloning into 'NeuroFetal-AI'...
remote: Enumerating objects: 2578, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 2578 (delta 108), reused 119 (delta 90), pack-reused 2417 (from 3)
Receiving objects: 100% (2578/2578), 1.15 GiB | 36.43 MiB/s, done.
Resolving deltas: 100% (1466/1466), done.
Updating files: 100% (1257/1257), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/Krishna200608/NeuroFetal-AI
 * branch            main       -> FETCH_HEAD
Already up to date.
✓ Cloned and checked out main!


### 1.5 Git Credentials

In [4]:
!git config --global user.email "krishnasikheriya001@gmail.com"
!git config --global user.name "Krishna200608"
print("✓ Git credentials set.")

✓ Git credentials set.


### 1.6 Install Dependencies

In [5]:
print("Installing libraries...")
!pip install -q wfdb shap scipy imbalanced-learn pyngrok filterpy \
    scikit-learn matplotlib seaborn pandas numpy tensorflow \
    streamlit plotly python-dotenv xgboost lightgbm
print("✓ Dependencies installed.")

Installing libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 110.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
✓ Dependencies installed.


---
## 2. Data Ingestion

Processes raw `.dat`/`.hea` files into clean `.npy` arrays (18 features, pH 7.15, quality filter).

In [6]:
!python Code/scripts/data_ingestion.py

Found 552 records.
pH threshold: 7.15
Max signal loss: 50%
Processed 100 records...
Processed 200 records...
Processed 300 records...
Processed 400 records...
Processed 500 records...

Processing complete.
  Patients: 552
  Total windows: 2546
  Skipped (quality): 214
  Shapes: X_fhr=(2546, 1200), X_uc=(2546, 1200), X_tabular=(2546, 18), y=(2546,)
  Tabular features (18): ['Age', 'Parity', 'Gestation', 'Gravidity', 'Weight', 'fhr_baseline', 'fhr_stv', 'fhr_ltv', 'fhr_accel_count', 'fhr_decel_count', 'fhr_decel_area', 'fhr_range', 'fhr_iqr', 'fhr_entropy', 'uc_freq', 'uc_intensity_mean', 'fhr_uc_lag', 'signal_loss_pct']
  Class balance: 470.0 compromised / 2546 total (18.5%)


---
## 3. Self-Supervised Pretraining

Train the Masked Autoencoder (MAE) on FHR data.
Saves encoder weights → `Code/models/pretrained_fhr_encoder.weights.keras`

In [7]:
!python Code/scripts/pretrain.py

2026-02-27 05:32:59.539845: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772170379.584410    2124 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772170379.597559    2124 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772170379.633518    2124 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772170379.633549    2124 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772170379.633556    2124 computation_placer.cc:177] computation placer alr

---
## 4. Primary Model Training (Focal Loss γ=1.5)

Train **AttentionFusionResNet** with 5-Fold CV + TimeGAN augmentation.

**V5.0 change:** Focal Loss γ reduced from 2.0 → 1.5 to reduce overconfidence.

In [8]:
# Pull latest changes (includes calibration fixes)
!git pull origin main

From https://github.com/Krishna200608/NeuroFetal-AI
 * branch            main       -> FETCH_HEAD
Already up to date.


In [9]:
!python Code/scripts/train.py

2026-02-27 05:35:42.725635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772170542.746991    3326 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772170542.754204    3326 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772170542.770587    3326 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772170542.770611    3326 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772170542.770615    3326 computation_placer.cc:177] computation placer alr

---
## 5. Diverse Ensemble Training + Isotonic Calibration

Trains 3 diverse models and stacks with **CalibratedClassifierCV (Isotonic Regression)**:
- Model A: AttentionFusionResNet (from Step 4)
- Model B: 1D-InceptionNet
- Model C: XGBoost

The meta-learner is now wrapped with `CalibratedClassifierCV(method='isotonic', cv=5)` for
perfectly calibrated output probabilities.

**Outputs:**
- `Code/models/inception_model_fold_*.keras`
- `Code/models/xgboost_model_fold_*.pkl`
- `Code/models/stacking_meta_learner.pkl` (calibrated)

In [10]:
!python Code/scripts/train_diverse_ensemble.py

NeuroFetal AI — Diverse Ensemble Training (Phase 5)

Data: FHR=(2546, 1200, 1), Tab=(2546, 18), y=(2546,)
Class balance: 18.5% positive
2026-02-27 07:37:04.251946: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772177824.272359   45202 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772177824.279340   45202 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772177824.295979   45202 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772177824.296005   45202 computation_placer.cc:177] computation placer already registered. Please check linkage a

---
## 6. Uncertainty Evaluation (Ensemble Variance)

Evaluates uncertainty using **ensemble disagreement** (std-dev across 3 base models)
instead of the old MC Dropout approach.

**Outputs:**
- Calibration curves per fold
- Uncertainty histograms per fold
- Expected Calibration Error (ECE)

In [11]:
!python Code/scripts/evaluate_uncertainty.py

2026-02-27 07:48:04.045257: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772178484.067397   54699 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772178484.074289   54699 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772178484.090608   54699 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772178484.090634   54699 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772178484.090638   54699 computation_placer.cc:177] computation placer alr

---
## 7. Push Trained Models to GitHub

Commit all trained models and evaluation reports back to the repository.

In [12]:
!git add Code/models/*.keras Code/models/*.pkl Code/models/*.npy
!git add Reports/uncertainty_analysis/ -f
!git commit -m "V5.0: Retrained ensemble with Isotonic calibration + γ=1.5"
!git push origin main
print("✓ Models and reports pushed to GitHub!")

[main 8eeea58] V5.0: Retrained ensemble with Isotonic calibration + γ=1.5
 29 files changed, 0 insertions(+), 0 deletions(-)
 rewrite Code/models/oof_predictions.npy (98%)
 rewrite Code/models/stacking_meta_learner.pkl (82%)
 rewrite Code/models/xgboost_model_fold_1.pkl (67%)
 rewrite Code/models/xgboost_model_fold_2.pkl (65%)
 rewrite Code/models/xgboost_model_fold_3.pkl (65%)
 rewrite Code/models/xgboost_model_fold_4.pkl (67%)
 rewrite Code/models/xgboost_model_fold_5.pkl (65%)
 rewrite Reports/uncertainty_analysis/fold_1/calibration_curve.png (98%)
 rewrite Reports/uncertainty_analysis/fold_1/uncertainty_histogram.png (98%)
 rewrite Reports/uncertainty_analysis/fold_2/calibration_curve.png (98%)
 rewrite Reports/uncertainty_analysis/fold_2/uncertainty_histogram.png (99%)
 rewrite Reports/uncertainty_analysis/fold_3/calibration_curve.png (98%)
 rewrite Reports/uncertainty_analysis/fold_3/uncertainty_histogram.png (99%)
 rewrite Reports/uncertainty_analysis/fold_4/calibration_curve.pn

---
## ✅ Done!

After this notebook completes:
1. All 3 ensemble models are retrained with reduced Focal Loss γ
2. The stacking meta-learner is calibrated with Isotonic Regression
3. Uncertainty is computed via ensemble variance (not MC Dropout)
4. Models are pushed to GitHub — pull locally and run the Streamlit app to verify